# Moshi Compression — Student Web Demo (ngrok)

**Goal.** Run the official Moshi web UI against the compressed STUDENT model
so you can have a live voice conversation with it in a browser — same UX as
the teacher, but powered by your Phase-2/Phase-3 SmolLM2 temporal transformer.

**What this does.**
1. Load the student checkpoint into a full `LMModel` shell (same as the offline
   demo notebook).
2. Instantiate Moshi's own `ServerState` around the student model — aiohttp
   websocket endpoint `/api/chat`, Opus streaming, text captions, all the
   official UX.
3. Download the official web UI (`dist.tgz` from `kyutai/moshi-artifacts`) and
   serve it as static files.
4. Start the aiohttp server on a background thread.
5. Open an ngrok tunnel to port 8998 and print the public URL.

**How to use.**
1. Run every cell top-to-bottom.
2. After the last cell prints a `https://...ngrok...` URL, open it in a
   browser (Chrome/Edge/Firefox, microphone permission required).
3. Click "Connect" in the web UI and start talking.

**Caveats for your demo.**
- First response may be laggy (~1-2 s). The warmup cell tries to prime the
  model to avoid first-frame latency spikes.
- Browser MUST use HTTPS for microphone access. ngrok gives you HTTPS free.
- The Phase-2 student has NOT had its Depformer unfrozen yet — audio quality
  is an upper bound of the frozen Depformer's ability to turn Phase-1/2
  hidden states into codes. Expect babble-level or proto-speech output.

**Datasets required**:
- `mhassann/moshi-p2-ckpt` (or `mhassann/moshi-p3-ckpt` if done)
- `tasfiatanha/moshi-frozen-heads`
- `tasfiatanha/moshi-repo`


## Cell 1 — Global patches + Kaggle env checks

In [2]:
import os, sys
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch._dynamo.config.disable = True
# sys.stdout.reconfigure(encoding="utf-8")
print("torch.compile disabled, expandable_segments enabled")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, {p.total_memory/1e9:.1f} GB")

# Kaggle notebooks DO allow outbound connections while the notebook is running
# (Internet must be toggled ON in the notebook settings — right sidebar).
import urllib.request
try:
    urllib.request.urlopen("https://api.ngrok.com", timeout=3)
    print("Internet: OK")
except Exception as e:
    print(f"Internet: FAILED ({e}). Enable Internet in the Kaggle notebook sidebar.")


torch.compile disabled, expandable_segments enabled
  cuda:0 = Tesla T4, 15.6 GB
  cuda:1 = Tesla T4, 15.6 GB
Internet: FAILED (HTTP Error 400: Bad Request). Enable Internet in the Kaggle notebook sidebar.


## Cell 2 — Installs (moshi + server deps + pyngrok)

In [3]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "einops",
    "safetensors",
    "sphn",
    "aiohttp",
    "pyngrok",
    "huggingface_hub",
    "nest_asyncio",  # so we can run asyncio in a Jupyter cell
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes
print(f"moshi from: {moshi.__file__}")

# Patch CUDAGraphed (must come AFTER moshi import so the class is resolvable)
import moshi.utils.compile as _moshi_compile
class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)
_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed patched → no-op")
print("=== Cell 2 PASSED ===")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 68.2 MB/s eta 0:00:00
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
CUDAGraphed patched → no-op
=== Cell 2 PASSED ===


## Cell 3 — Write smol_temporal.py

Same wrapper as training / offline demo.


In [4]:
import pathlib

DST = pathlib.Path('/kaggle/working/moshi_repo/moshi/models/smol_temporal.py')

_SRC = '''# moshi/models/smol_temporal.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import torch
import torch.nn as nn
import transformers
from ..modules.streaming import StreamingModule, State


@dataclass
class _SmolState(State):
    past_key_values: Optional[tuple] = field(default=None)

    def reset(self, reset_mask: torch.Tensor) -> None:
        super().reset(reset_mask)
        self.past_key_values = None


class SmolTemporalTransformer(StreamingModule[_SmolState]):
    def __init__(
        self,
        teacher_dim: int = 4096,
        student_dim: int = 2048,
        hf_name: str = "HuggingFaceTB/SmolLM2-1.7B",
        rope_theta: float = 10_000.0,
        device: str = "cuda:0",
        dtype: torch.dtype = torch.float16,
    ):
        super().__init__()
        self.teacher_dim = teacher_dim
        self.student_dim = student_dim

        cfg = transformers.AutoConfig.from_pretrained(hf_name)
        cfg.rope_theta = rope_theta
        cfg.use_cache = True
        cfg.attn_implementation = "eager"
        self.backbone = transformers.AutoModel.from_pretrained(
            hf_name, config=cfg, torch_dtype=dtype,
        )
        if hasattr(self.backbone, "embed_tokens"):
            self.backbone.embed_tokens = nn.Identity()

        self.in_adapter  = nn.Linear(teacher_dim, student_dim, bias=False)
        self.out_adapter = nn.Linear(student_dim, teacher_dim, bias=False)
        nn.init.normal_(self.in_adapter.weight,  std=1.0 / (teacher_dim ** 0.5))
        nn.init.normal_(self.out_adapter.weight, std=1.0 / (student_dim ** 0.5))

        self.to(device=device, dtype=dtype)

    def _init_streaming_state(self, batch_size: int) -> _SmolState:
        device = self.in_adapter.weight.device
        return _SmolState(batch_size=batch_size, device=device, past_key_values=None)

    def forward(
        self,
        x: torch.Tensor,
        cross_attention_src: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        assert cross_attention_src is None
        assert x.dim() == 3 and x.shape[-1] == self.teacher_dim

        x = x.to(self.in_adapter.weight.device)

        past_kv = (self._streaming_state.past_key_values
                   if self._streaming_state is not None else None)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            h = self.in_adapter(x)
            out = self.backbone(
                inputs_embeds=h,
                past_key_values=past_kv,
                use_cache=(self._streaming_state is not None),
                return_dict=True,
            )
            y = self.out_adapter(out.last_hidden_state)

        if self._streaming_state is not None:
            self._streaming_state.past_key_values = out.past_key_values

        return y

    def student_state_dict(self):
        return {
            "backbone":    self.backbone.state_dict(),
            "in_adapter":  self.in_adapter.state_dict(),
            "out_adapter": self.out_adapter.state_dict(),
        }

    def load_student_state_dict(self, sd: dict):
        self.backbone.load_state_dict(sd["backbone"])
        self.in_adapter.load_state_dict(sd["in_adapter"])
        self.out_adapter.load_state_dict(sd["out_adapter"])
'''

DST.write_text(_SRC)
print(f"Wrote {DST} ({DST.stat().st_size} bytes)")

import importlib, moshi.models
if hasattr(moshi.models, "smol_temporal"):
    importlib.reload(moshi.models.smol_temporal)
from moshi.models.smol_temporal import SmolTemporalTransformer
print("=== Cell 3 PASSED ===")


Wrote /kaggle/working/moshi_repo/moshi/models/smol_temporal.py (3255 bytes)
=== Cell 3 PASSED ===


## Cell 4 — Build student LMModel + load checkpoint

Single-device (cuda:0) layout for inference simplicity. Identical to the
offline demo's Cell 6.


In [5]:
import torch, pathlib, gc, glob, os
import pickle as _pickle
from moshi.models.lm import LMModel
from moshi.models.smol_temporal import SmolTemporalTransformer


class _NumpyCompatUnpickler(_pickle.Unpickler):
    _REMAP = {
        "numpy.core.multiarray": "numpy._core.multiarray",
        "numpy.core.numeric":    "numpy._core.numeric",
        "numpy.core.umath":      "numpy._core.umath",
        "numpy.core":            "numpy._core",
    }
    def find_class(self, module, name):
        return super().find_class(self._REMAP.get(module, module), name)

class _NpPickle:
    Unpickler        = _NumpyCompatUnpickler
    loads            = staticmethod(_pickle.loads)
    load             = staticmethod(_pickle.load)
    dump             = staticmethod(_pickle.dump)
    dumps            = staticmethod(_pickle.dumps)
    HIGHEST_PROTOCOL = _pickle.HIGHEST_PROTOCOL
    DEFAULT_PROTOCOL = _pickle.DEFAULT_PROTOCOL
    PickleError      = _pickle.PickleError
    UnpicklingError  = _pickle.UnpicklingError

def _torch_load(path, **kw):
    kw.setdefault("weights_only", False)
    kw.setdefault("pickle_module", _NpPickle)
    return torch.load(path, **kw)


print("Building student LMModel shell ...")
student_lm = LMModel(
    dim=4096, num_heads=32, num_layers=32, hidden_scale=4.125,
    gating="silu", norm="rms_norm_f32", positional_embedding="rope",
    context=300,  n_q=16, dep_q=8, card=2048, text_card=32000,  # 300 frames=24s; 3000 OOMs on T4
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    depformer_dim=1024, depformer_dim_feedforward=4224,
    depformer_num_heads=16, depformer_num_layers=6,
    depformer_multi_linear=True, depformer_weights_per_step=True,
    depformer_context=8, depformer_pos_emb="none",
    existing_text_padding_id=3,
).to(dtype=torch.float16)

smol_tt = SmolTemporalTransformer(
    teacher_dim=4096, student_dim=2048,
    hf_name="HuggingFaceTB/SmolLM2-1.7B",
    rope_theta=10_000.0, device="cpu", dtype=torch.float16,
)
student_lm.transformer = smol_tt
gc.collect(); torch.cuda.empty_cache()

# Frozen heads
FROZEN_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-frozen-heads"),
    pathlib.Path("/kaggle/input/moshi-frozen-heads"),
]
frozen_dir = None
for p in FROZEN_CANDIDATES:
    if p.exists():
        frozen_dir = p
        break
if frozen_dir is None:
    raise FileNotFoundError(f"Frozen heads not found: {FROZEN_CANDIDATES}")

print(f"Loading frozen heads from {frozen_dir} ...")
frozen_modules = [
    "emb", "text_emb", "out_norm", "text_linear",
    "depformer_in", "depformer",
    "depformer_emb", "depformer_text_emb", "linears",
]
for name in frozen_modules:
    pt_file = frozen_dir / f"{name}.pt"
    if pt_file.exists():
        sd = torch.load(pt_file, map_location="cpu", weights_only=True)
        getattr(student_lm, name).load_state_dict(sd)

# Student checkpoint (prefer P3 if available, else P2)
CKPT_CANDIDATES = [
    "/kaggle/input/datasets/mhassann/moshi-p3-ckpt",
    "/kaggle/input/moshi-p3-ckpt",
    "/kaggle/input/datasets/mhassann/moshi-p2-ckpt",
    "/kaggle/input/moshi-p2-ckpt",
]
student_ckpt = None
for d in CKPT_CANDIDATES:
    if os.path.isdir(d):
        c = sorted(glob.glob(f"{d}/ckpt_step_*.pt"))
        if c:
            student_ckpt = c[-1]
            break
if student_ckpt is None:
    raise FileNotFoundError("No P2/P3 checkpoint found")

print(f"Loading student weights from {student_ckpt} ...")
ckpt = _torch_load(student_ckpt, map_location="cpu")
ckpt_phase = ckpt.get("phase", "?")
smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
del ckpt
gc.collect(); torch.cuda.empty_cache()
print(f"Student loaded: phase {ckpt_phase}")

# Move to cuda:0 (single-device inference)
student_lm.transformer.to("cuda:0")
for attr in ["emb", "text_emb", "out_norm", "text_linear", "depformer_in",
             "depformer", "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:0")

# Single-device forward_text (no cross-device hops needed when everything is on cuda:0)
import types as _types
def _single_device_forward_text(self, sequence, sum_condition=None, cross_attention_src=None):
    B, K, S = sequence.shape
    device = next(self.emb[0].parameters()).device
    input_sequence = sequence.to(device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    if sum_condition is not None:
        input_ = input_ + sum_condition.to(input_)
    if cross_attention_src is not None:
        cross_attention_src = cross_attention_src.to(input_)

    _tt_dtype = next(self.transformer.parameters()).dtype
    transformer_out = self.transformer(
        input_.to(dtype=_tt_dtype), cross_attention_src=cross_attention_src
    )

    if self.out_norm:
        _on = next(self.out_norm.parameters())
        transformer_out = self.out_norm(transformer_out.to(dtype=_on.dtype))
    _tl = next(self.text_linear.parameters())
    text_logits = self.text_linear(transformer_out.to(dtype=_tl.dtype))
    text_logits = text_logits[:, None]
    return transformer_out, text_logits

student_lm.forward_text = _types.MethodType(_single_device_forward_text, student_lm)
# The `device` attribute is read by LMGen / ServerState
student_lm.__dict__["device"] = torch.device("cuda:0")  # property has no setter

student_lm.eval()
for p in student_lm.parameters():
    p.requires_grad_(False)

torch.cuda.synchronize()
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} free {free/1e9:.2f} / {total/1e9:.2f} GB")
print(f"Student phase: {ckpt_phase}")
print("=== Cell 4 PASSED ===")


Building student LMModel shell ...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading frozen heads from /kaggle/input/datasets/tasfiatanha/moshi-frozen-heads ...
Loading student weights from /kaggle/input/datasets/mhassann/moshi-p3-ckpt/ckpt_step_1056.pt ...
Student loaded: phase P3
cuda:0 free 10.03 / 15.64 GB
cuda:1 free 15.53 / 15.64 GB
Student phase: P3
=== Cell 4 PASSED ===


## Cell 5 — Load Mimi + text tokenizer

These are identical to the teacher — the student's Mimi and tokenizer are
unchanged through Phases 1-3.


In [6]:
import torch
from moshi.models.loaders import get_mimi
from huggingface_hub import hf_hub_download
import sentencepiece

print("Fetching Mimi + tokenizer from HF ...")
mimi_path = hf_hub_download(
    repo_id="kyutai/moshiko-pytorch-bf16",
    filename="tokenizer-e351c8d8-checkpoint125.safetensors",
    cache_dir="/tmp/mimi",
)
# Keep Mimi in fp32. Its conv layers reject fp32 PCM input if Mimi itself is
# fp16 (RuntimeError: Input type (float) and bias type (c10::Half) should be
# the same). Mimi is ~85 MB — the fp16 savings aren't worth the cast plumbing.
mimi = get_mimi(mimi_path, device="cuda:0")
mimi.eval()
print(f"Mimi loaded (fp32): sample_rate={mimi.sample_rate}, frame_rate={mimi.frame_rate}")

tok_path = hf_hub_download(
    repo_id="kyutai/moshiko-pytorch-bf16",
    filename="tokenizer_spm_32k_3.model",
    cache_dir="/tmp/mimi",
)
text_tokenizer = sentencepiece.SentencePieceProcessor()
text_tokenizer.Load(tok_path)
print(f"Tokenizer vocab: {text_tokenizer.vocab_size()}")
print("=== Cell 5 PASSED ===")


Fetching Mimi + tokenizer from HF ...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

Mimi loaded (fp32): sample_rate=24000, frame_rate=12.5


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

Tokenizer vocab: 32000
=== Cell 5 PASSED ===


## Cell 6 — Build ServerState + warmup

`ServerState` is Moshi's own class from `moshi/server.py`. We construct it
manually with our student `LMModel` instead of going through the CLI. The
`warmup()` method runs a few dummy frames to prime CUDA / the streaming state
so the first real connection doesn't have a ~1 s spike.


In [7]:
import asyncio, gc, time
import numpy as np
import aiohttp
from aiohttp import web
import sphn
import torch
from moshi.models.lm import LMGen
from moshi.run_inference import get_condition_tensors
from moshi.client_utils import log

# IMPORTANT: we deliberately do NOT `from moshi.server import ServerState`.
# moshi/server.py ends with `with torch.no_grad(): main()` at module scope —
# importing it runs the full CLI, which downloads and loads the TEACHER onto
# cuda:0 → OOM. We inline ServerState here verbatim instead.

def _mem(tag):
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info(0)
    used = (total - free) / 1e9
    print(f"[MEM {tag:28s}] used={used:5.2f}  free={free/1e9:5.2f} / {total/1e9:5.2f} GB")

gc.collect(); torch.cuda.empty_cache()
_mem("before ServerState")


class ServerState:
    """Inlined from moshi/server.py — same behavior, minus the CLI-side
    module-level teacher load that was OOMing us.

    Added vs upstream: explicit `torch.no_grad()` around every inference path
    (mimi.encode / lm_gen.step / mimi.decode). Upstream uses CUDAGraphed which
    implicitly disables grad; we patched CUDAGraphed to a no-op in Cell 2 so
    we have to enforce no_grad manually."""

    def __init__(self, model_type, mimi, text_tokenizer, lm, cfg_coef, device, **kwargs):
        self.model_type = model_type
        self.mimi = mimi
        self.text_tokenizer = text_tokenizer
        condition_tensors = get_condition_tensors(model_type, lm, batch_size=1, cfg_coef=cfg_coef)
        self.lm_gen = LMGen(lm, cfg_coef=cfg_coef, condition_tensors=condition_tensors, **kwargs)

        self.device = device
        self.frame_size = int(self.mimi.sample_rate / self.mimi.frame_rate)
        self.lock = asyncio.Lock()

        self.mimi.streaming_forever(1)
        self.lm_gen.streaming_forever(1)

    def warmup(self):
        with torch.no_grad():
            for _ in range(4):
                chunk = torch.zeros(1, 1, self.frame_size, dtype=torch.float32, device=self.device)
                codes = self.mimi.encode(chunk)
                for c in range(codes.shape[-1]):
                    tokens = self.lm_gen.step(codes[:, :, c: c + 1])
                    if tokens is None:
                        continue
                    _ = self.mimi.decode(tokens[:, 1:])
            torch.cuda.synchronize()

    async def decode_and_send(self, tokens, ws, opus_writer):
        assert tokens.shape[1] == self.lm_gen.lm_model.dep_q + 1
        with torch.no_grad():
            main_pcm = self.mimi.decode(tokens[:, 1:]).detach().cpu()
        opus_bytes = opus_writer.append_pcm(main_pcm[0, 0].numpy())
        if len(opus_bytes) > 0:
            await ws.send_bytes(b"\x01" + opus_bytes)
        text_token = tokens[0, 0, 0].item()
        if text_token not in (0, 3):
            _text = self.text_tokenizer.id_to_piece(text_token)
            _text = _text.replace("▁", " ")
            await ws.send_bytes(b"\x02" + bytes(_text, encoding="utf8"))
            log("info", f"text token '{_text}'")

    async def recv_loop(self, ws, opus_reader, opus_writer):
        all_pcm_data = None
        skip_frames = 1
        try:
            async for message in ws:
                if message.type == aiohttp.WSMsgType.ERROR:
                    log("error", f"{ws.exception()}"); break
                elif message.type == aiohttp.WSMsgType.CLOSED:
                    break
                elif message.type != aiohttp.WSMsgType.BINARY:
                    log("error", f"unexpected message type {message.type}"); continue
                data = message.data
                if not isinstance(data, bytes) or len(data) == 0:
                    continue
                kind = data[0]
                if kind == 1:  # audio
                    pcm = opus_reader.append_bytes(data[1:])
                    if pcm.shape[-1] == 0:
                        continue
                    all_pcm_data = pcm if all_pcm_data is None else np.concatenate((all_pcm_data, pcm))
                    while all_pcm_data.shape[-1] >= self.frame_size:
                        be = time.time()
                        chunk = all_pcm_data[: self.frame_size]
                        all_pcm_data = all_pcm_data[self.frame_size:]
                        chunk = torch.from_numpy(chunk).to(device=self.device)[None, None]
                        with torch.no_grad():
                            codes = self.mimi.encode(chunk)
                        if skip_frames:
                            self.mimi.reset_streaming()
                            skip_frames -= 1
                        for c in range(codes.shape[-1]):
                            with torch.no_grad():
                                tokens = self.lm_gen.step(codes[:, :, c: c + 1])
                            if tokens is None:
                                continue
                            await self.decode_and_send(tokens, ws, opus_writer)
                        log("info", f"frame handled in {1000 * (time.time() - be):.1f}ms")
                else:
                    log("warning", f"unknown message kind {kind}")
        finally:
            log("info", "connection closed")

    async def handle_chat(self, request):
        ws = web.WebSocketResponse()
        await ws.prepare(request)
        log("info", "accepted connection")
        async with self.lock:
            opus_writer = sphn.OpusStreamWriter(self.mimi.sample_rate)
            opus_reader = sphn.OpusStreamReader(self.mimi.sample_rate)
            self.mimi.reset_streaming()
            self.lm_gen.reset_streaming()
            await ws.send_bytes(b"\x00")  # handshake
            await self.recv_loop(ws, opus_reader, opus_writer)
        log("info", "done with connection")
        return ws


# Defensive: eval mode (LMGen asserts not training).
student_lm.eval()
for _sub in student_lm.modules():
    _sub.eval()
assert not student_lm.training, "student_lm must be in eval mode"
print(f"student_lm.training = {student_lm.training}")

# Clear any leftover streaming state from a prior Cell 6 run. This is the fix
# for `AssertionError: <name> is already streaming!` — reset_streaming()
# only clears buffers, it does NOT set `_streaming_state = None`. We must
# call `_stop_streaming()` to clear the flag, otherwise the next
# `streaming_forever(1)` inside ServerState.__init__ will trip the assert.
def _full_stop_stream(mod):
    try:
        mod._stop_streaming()
        mod._cached_children = None  # force re-scan on next streaming()
    except Exception as e:
        print(f"  _stop_streaming on {type(mod).__name__}: {e!r}")

_full_stop_stream(mimi)
_full_stop_stream(student_lm)
# Also kill any previously-constructed lm_gen object that might still hold state
try:
    state.lm_gen._stop_streaming()
    state.lm_gen._cached_children = None
except NameError:
    pass
except Exception as e:
    print(f"  prior lm_gen stop: {e!r}")

gc.collect(); torch.cuda.empty_cache()
_mem("after stream reset")

state = ServerState(
    model_type="moshi",
    mimi=mimi,
    text_tokenizer=text_tokenizer,
    lm=student_lm,
    cfg_coef=1.0,
    device="cuda:0",
)
_mem("after ServerState built")
print("ServerState constructed.")

print("Skipping warmup to preserve GPU memory (T4 constraint).")
_mem("end of Cell 6")
print("=== Cell 6 PASSED ===")


[MEM before ServerState          ] used= 6.00  free= 9.64 / 15.64 GB
student_lm.training = False
[MEM after stream reset          ] used= 6.00  free= 9.64 / 15.64 GB
[MEM after ServerState built     ] used= 6.02  free= 9.62 / 15.64 GB
ServerState constructed.
Skipping warmup to preserve GPU memory (T4 constraint).
[MEM end of Cell 6               ] used= 6.02  free= 9.62 / 15.64 GB
=== Cell 6 PASSED ===


## Cell 7 — Download official Moshi web UI (dist.tgz)

Fetches the pre-built React UI that the Kyutai server normally serves. This
is the same UI you'd get at moshi.chat, just pointed at your tunnel URL.


In [8]:
import tarfile, pathlib
from huggingface_hub import hf_hub_download

print("Downloading dist.tgz from kyutai/moshi-artifacts ...")
dist_tgz = hf_hub_download("kyutai/moshi-artifacts", "dist.tgz",
                           cache_dir="/tmp/moshi_dist")
dist_tgz = pathlib.Path(dist_tgz)
dist = dist_tgz.parent / "dist"
if not dist.exists():
    with tarfile.open(dist_tgz, "r:gz") as tar:
        tar.extractall(path=dist_tgz.parent)

index_html = dist / "index.html"
assert index_html.exists(), f"dist layout unexpected: no index.html in {dist}"
print(f"Web UI extracted: {dist}")
print(f"  index.html: {index_html.stat().st_size} bytes")
STATIC_PATH = str(dist)
print("=== Cell 7 PASSED ===")


dist.tgz:   0%|          | 0.00/589k [00:00<?, ?B/s]

Web UI extracted: /tmp/moshi_dist/models--kyutai--moshi-artifacts/snapshots/5040b2bc0ede3531913ce11bf591e7c822164a54/dist
  index.html: 465 bytes
=== Cell 7 PASSED ===


/tmp/ipykernel_55/2291975703.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=dist_tgz.parent)


## Cell 8 — Build the aiohttp app

Same routing as `moshi/server.py`:
  - `/api/chat` websocket (binary-framed audio in, Opus audio + text tokens out)
  - `/` serves `index.html`
  - `/*` serves static assets (JS, CSS, wasm worklet)


In [9]:
import os, asyncio
from aiohttp import web

# aiohttp Application
app = web.Application()
app.router.add_get("/api/chat", state.handle_chat)

async def handle_root(_):
    return web.FileResponse(os.path.join(STATIC_PATH, "index.html"))

app.router.add_get("/", handle_root)
app.router.add_static(
    "/", path=STATIC_PATH, follow_symlinks=True, name="static"
)
print(f"aiohttp app wired: / → index.html, /api/chat → ServerState.handle_chat, /* → static")
print("=== Cell 8 PASSED ===")


aiohttp app wired: / → index.html, /api/chat → ServerState.handle_chat, /* → static
=== Cell 8 PASSED ===


## Cell 9 — Start the server on a background thread

We use `threading.Thread` + its own asyncio loop so the Jupyter cell returns
immediately (web.run_app would otherwise block the kernel forever). The
aiohttp runner listens on 127.0.0.1:8998; ngrok will forward to this.


In [10]:
import threading, asyncio
from aiohttp import web

PORT = 8998

# We need a sentinel so we can shut the server down later if the user re-runs
# this cell (Kaggle notebooks re-execute cells without restarting the kernel).
def _start_server_in_thread(app, port):
    loop = asyncio.new_event_loop()

    async def _run():
        runner = web.AppRunner(app)
        await runner.setup()
        site = web.TCPSite(runner, host="127.0.0.1", port=port)
        await site.start()
        print(f"aiohttp server listening on http://127.0.0.1:{port}")
        # keep the thread alive
        while True:
            await asyncio.sleep(3600)

    def _thread_main():
        asyncio.set_event_loop(loop)
        loop.run_until_complete(_run())

    t = threading.Thread(target=_thread_main, name="moshi-aiohttp", daemon=True)
    t.start()
    return t, loop

# Kill any existing server thread from a prior run of this cell
existing = [t for t in threading.enumerate() if t.name == "moshi-aiohttp"]
if existing:
    print(f"Found {len(existing)} existing server thread(s); they are daemons "
          f"and will be abandoned (you may see address-in-use below).")

server_thread, server_loop = _start_server_in_thread(app, PORT)

# Give it ~1 s to bind
import time
time.sleep(1.0)

# Sanity probe: can we connect locally?
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(2)
try:
    s.connect(("127.0.0.1", PORT))
    s.close()
    print(f"Local probe OK: 127.0.0.1:{PORT} is reachable")
except Exception as e:
    print(f"Local probe FAILED: {e}")
print("=== Cell 9 PASSED ===")


aiohttp server listening on http://127.0.0.1:8998
Local probe OK: 127.0.0.1:8998 is reachable
=== Cell 9 PASSED ===


## Cell 10 — Open ngrok tunnel and print the public URL

If you have an ngrok account, paste your authtoken into the NGROK_AUTHTOKEN
cell below (recommended — gives you a stable session and higher limits). The
free, anonymous tunnel works too but the URL changes each time and the
session lasts ~2 h.

Get your authtoken (free) at https://dashboard.ngrok.com/get-started/your-authtoken


In [12]:
import os
from pyngrok import ngrok, conf

# === SET THIS IF YOU HAVE AN NGROK ACCOUNT (free) ===
# Either paste the token below, or set NGROK_AUTHTOKEN as a Kaggle secret.
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "3CWSLkdHQCH4v36I6P7Ij9WPCN2_2zJax26CnZJJAP5iRZGCV")  # e.g. "2abcXYZ..."

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    print("ngrok authtoken set (authenticated tunnel).")
else:
    print("No NGROK_AUTHTOKEN set — using anonymous tunnel (2 h limit).")

# Kill any prior tunnels from re-runs
for t in ngrok.get_tunnels():
    print(f"Killing existing tunnel: {t.public_url}")
    ngrok.disconnect(t.public_url)

# Open tunnel to our local server
public_tunnel = ngrok.connect(PORT, "http", bind_tls=True)
public_url = public_tunnel.public_url
# ngrok returns http:// or https:// depending on version; force https
if public_url.startswith("http://"):
    public_url = "https://" + public_url[len("http://"):]

print()
print("=" * 60)
print("STUDENT WEB UI READY")
print("=" * 60)
print(f"Open this URL in Chrome/Edge/Firefox:")
print(f"  {public_url}")
print()
print("Click Connect, allow microphone access, and start talking.")
print(f"(Local server: 127.0.0.1:{PORT} · Tunnel: {public_tunnel.public_url})")
print("=" * 60)
print()
print("To stop the tunnel, run Cell 11 below. To stop the server, interrupt")
print("the kernel (Runtime → Interrupt).")


ngrok authtoken set (authenticated tunnel).
                                                                                                    
STUDENT WEB UI READY
Open this URL in Chrome/Edge/Firefox:
  https://deserve-backing-mace.ngrok-free.dev

Click Connect, allow microphone access, and start talking.
(Local server: 127.0.0.1:8998 · Tunnel: https://deserve-backing-mace.ngrok-free.dev)

To stop the tunnel, run Cell 11 below. To stop the server, interrupt
the kernel (Runtime → Interrupt).
[Info] accepted connection


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


[Info] frame handled in 1906.0ms
[Info] text token ' rep'
[Info] frame handled in 1025.7ms
[Info] text token ' belt'
[Info] frame handled in 162.8ms
[Info] text token '<0x14>'
[Info] frame handled in 68.6ms
[Info] text token '<0x14>'
[Info] frame handled in 53.3ms
[Info] text token '<0x14>'
[Info] frame handled in 53.4ms
[Info] text token '<0x14>'
[Info] frame handled in 53.1ms
[Info] text token '<0x14>'
[Info] frame handled in 52.8ms
[Info] text token '<0x14>'
[Info] frame handled in 50.7ms
[Info] text token '<0x14>'
[Info] frame handled in 49.2ms
[Info] text token '<0x14>'
[Info] frame handled in 48.7ms
[Info] text token '<0x14>'
[Info] frame handled in 48.8ms
[Info] text token '<0x14>'
[Info] frame handled in 50.6ms
[Info] text token '<0x14>'
[Info] frame handled in 49.9ms
[Info] text token '<0x14>'
[Info] frame handled in 49.4ms
[Info] text token '<0x14>'
[Info] frame handled in 49.1ms
[Info] text token '<0x14>'
[Info] frame handled in 49.5ms
[Info] text token '<0x14>'
[Info] frame

## Cell 11 — Stop the tunnel (run when done)

This closes the public URL but leaves the aiohttp server running locally
(so you can open a new tunnel later without rebuilding the model).


In [20]:
from pyngrok import ngrok

tunnels = ngrok.get_tunnels()
for t in tunnels:
    ngrok.disconnect(t.public_url)
    print(f"Closed: {t.public_url}")
if not tunnels:
    print("(no active tunnels)")


(no active tunnels)
